ШАГ 4: КЛАССИФИКАЦИЯ (IC50 > МЕДИАНЫ)

Целью данного этапа является построение модели бинарной классификации для предсказания превышения медианного значения признака. Решение этой задачи позволит отфильтровывать объекты с низкими значениями целевой переменной.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# и опять загружаем данные
df = pd.read_csv('cleaned_data.csv')

targets_to_exclude = ['IC50, mM', 'CC50, mM', 'SI', 'pIC50', 'pCC50', 'log_SI',
                      'IC50_above_med', 'CC50_above_med', 'SI_above_med', 'SI_above_8']
X = df.drop(columns=targets_to_exclude)
y = df['IC50_above_med']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

print("Данные загружены")
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}\n")

# пайплайны
pipelines_clf = {
    'LogisticRegression': Pipeline([('scaler', StandardScaler()),
                                    ('model', LogisticRegression(random_state=42, max_iter=1000))]),
    'SVC': Pipeline([('scaler', StandardScaler()),
                     ('model', SVC(random_state=42))]),
    'RandomForest': Pipeline([('scaler', StandardScaler()),
                              ('model', RandomForestClassifier(random_state=42))])
}

param_grids_clf = {
    'LogisticRegression': {
        'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'model__penalty': ['l2']
    },
    'SVC': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear']
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5]
    }
}

results_clf = []

for name in pipelines_clf:
    grid = GridSearchCV(pipelines_clf[name], param_grids_clf[name], cv=5,
                        scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    y_pred = grid.best_estimator_.predict(X_test)

    results_clf.append({
        'модель': name,
        'лучшие параметры': str(grid.best_params_),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results_clf).sort_values(by='Accuracy', ascending=False)
print("результаты классификации:")
print(results_df.to_string(index=False))

Данные загружены
Train: X=(800, 139), y=(800,)
Test:  X=(201, 139), y=(201,)

Обучение LogisticRegression...
Обучение SVC...
Обучение RandomForest...

Результаты классификации (IC50_above_med):
            Модель                                                                    Лучшие параметры  Accuracy  Precision  Recall  F1 Score
               SVC                                             {'model__C': 1, 'model__kernel': 'rbf'}  0.721393   0.692982    0.79  0.738318
      RandomForest {'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 200}  0.711443   0.675000    0.81  0.736364
LogisticRegression                                           {'model__C': 0.1, 'model__penalty': 'l2'}  0.671642   0.637097    0.79  0.705357


ВЫВОДЫ: 

Наилучшие результаты по совокупности метрик продемонстрировал метод опорных векторов (SVC) с нелинейным RBF-ядром, что указывает на эффективность нелинейного разделения классов в многомерном пространстве признаков. Случайный лес показал близкое значение точности и при этом обеспечил самую высокую полноту. Логистическая регрессия, как линейный классификатор, ожидаемо уступила нелинейным моделям, подтверждая сложную структуру данных, которую трудно описать простой разделяющей гиперплоскостью.

Для дальнейшего повышения качества классификации я предлагаю рассмотреть следующие направления. Во-первых, целесообразно применить более мощные алгоритмы, такие как градиентный бустинг (XGBoost, LightGBM) или полносвязные нейронные сети. Во-вторых, следует уделить внимание работе с признаками: текущий набор дескрипторов может содержать шум и сильно коррелирующие переменные; снижение размерности с помощью PCA или отбор по важности признаков помогут уменьшить переобучение. В-третьих, сбор дополнительных данных способен улучшить обобщающую способность моделей, поскольку имеющаяся разметка может быть зашумлена. Наконец, перспективным представляется применение стекового ансамблирования, комбинирующего предсказания случайного леса, метода опорных векторов и градиентного бустинга, что потенциально даст дополнительный прирост в качестве итоговой классификации.